# Lab 5 · Pre-training Objectives — MLM and CLM

> **Transformers teaching package — Lab.** Run top to bottom (Runtime → Run all). Read the notes, run the code,
> and finish the **Your turn 🧪** cell. Colors follow the *Neural Indigo* palette from the slides.


## What you'll build
The two self-supervised objectives that shape a model's personality:
**MLM** (BERT — fill masked blanks, bidirectional) and **CLM** (GPT — predict the next token, left-to-right).
You'll implement the data prep for each *and* watch real pretrained models do them.

In [ ]:
!pip install -q transformers torch
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoModelForCausalLM
print("ready")

## 1 · MLM data prep (what BERT trains on)
Take a sentence, randomly replace ~15% of tokens with `[MASK]`, and ask the model to recover them
using context from **both sides**.

In [ ]:
import random
tok = AutoTokenizer.from_pretrained("bert-base-uncased")

def make_mlm_example(text, p=0.15, seed=0):
    random.seed(seed)
    ids = tok(text)["input_ids"]
    labels = [-100]*len(ids)                       # -100 = "don't compute loss here"
    for i in range(1, len(ids)-1):                 # skip [CLS]/[SEP]
        if random.random() < p:
            labels[i] = ids[i]                     # remember the true token
            ids[i] = tok.mask_token_id             # corrupt the input
    return ids, labels

ids, labels = make_mlm_example("the quick brown fox jumps over the lazy dog", seed=3)
print("masked input :", tok.decode(ids))
print("targets      :", [tok.decode([l]) for l in labels if l != -100])

## 2 · Watch BERT fill the masks (bidirectional)

In [ ]:
mlm = AutoModelForMaskedLM.from_pretrained("bert-base-uncased").eval()

def fill(text, topk=5):
    enc = tok(text, return_tensors="pt")
    with torch.no_grad():
        logits = mlm(**enc).logits
    mask_pos = (enc.input_ids == tok.mask_token_id)[0].nonzero(as_tuple=True)[0]
    for pos in mask_pos:
        probs = logits[0, pos].softmax(-1)
        top = probs.topk(topk)
        guesses = [(tok.decode([i]).strip(), round(p.item(),3)) for p,i in zip(top.values, top.indices)]
        print(f"[MASK] @ {pos.item():>2} ->", guesses)

fill("the capital of france is [MASK].")
fill("i took my [MASK] to the vet because it was sick.")

## 3 · CLM data prep (what GPT trains on)
No masking — the label at each position is simply the **next** token. The model may only look left.

In [ ]:
gpt_tok = AutoTokenizer.from_pretrained("gpt2")
ids = gpt_tok("language models predict the next")["input_ids"]
inputs, targets = ids[:-1], ids[1:]
for a, b in zip(inputs, targets):
    print(f"given ...{gpt_tok.decode([a])!r:>14}  ->  predict {gpt_tok.decode([b])!r}")

## 4 · Watch GPT-2 predict the next token (causal)

In [ ]:
clm = AutoModelForCausalLM.from_pretrained("gpt2").eval()

def next_token(text, topk=5):
    enc = gpt_tok(text, return_tensors="pt")
    with torch.no_grad():
        logits = clm(**enc).logits
    probs = logits[0, -1].softmax(-1)
    top = probs.topk(topk)
    return [(gpt_tok.decode([i]).strip(), round(p.item(),3)) for p,i in zip(top.values, top.indices)]

for prompt in ["The capital of France is", "Once upon a", "2 + 2 ="]:
    print(f"{prompt!r:38} -> {next_token(prompt)}")

## 5 · The key difference — visibility
- **MLM / BERT:** every token attends **both ways** → deep understanding, but not a natural generator.
- **CLM / GPT:** every token attends **left only** → fluent generation.
- **T5 / span corruption:** mask whole spans and *generate* them with an encoder–decoder — a hybrid.

In [ ]:
# T5-style span corruption (schematic)
text = "the cat sat on the mat"
words = text.split()
# replace a span with a single sentinel; target = the removed span
corrupted = "the cat <extra_id_0> the mat"
target    = "<extra_id_0> sat on"
print("input :", corrupted)
print("target:", target)
print("\nT5 reads the corrupted text bidirectionally, then GENERATES the missing span.")

## Your turn 🧪
1. Increase the MLM masking probability `p` to 0.4 — the guesses get worse. Why is 15% a sweet spot?
2. Give GPT-2 a factual prompt and a creative prompt; compare how peaked the next-token distribution is.
3. Feed the same fill-in-the-blank sentence to **BERT** (`[MASK]`) and to **GPT-2** (as a prefix). Which is better at using the *right-hand* context, and why?

In [ ]:
# Your turn: compare left-vs-both-sided context
sentence_right_context = "the [MASK] barked loudly at the mailman."
print("BERT (sees both sides):")
fill(sentence_right_context)
print("\nGPT-2 only sees the left, so it cannot use 'barked' to guess the blank.")